### k-means

**Clustering** groups objects so that the objects in one group (cluster) are more similar to each other than to the objects in other groups. To make "similar" precise we need a **cost function**, so clustering is yet another optimization problem.

K-means is an **unsupervised** clustering algorithm: it needs no labels, only the data and the number of clusters $K$, which has to be known in advance. It looks for $K$ centroids (cluster centers) $c_k$ that minimize the sum of squared distances between every sample and the centroid of its cluster

$$ J = \sum_{k=1}^{K} \sum_{x \in \text{cluster } k} \lVert x - c_k \rVert^2 $$

The minimization is iterative. Starting from some initial centroids, we alternate two steps until nothing changes any more:

1. **Assignment** - assign every sample to its nearest centroid (nearest-neighbour clustering),
2. **Update** - move every centroid to the mean of the samples assigned to it.

Neither of the steps can increase $J$, so the algorithm always converges. It converges only to a *local* minimum, though: where it ends depends on where it started (we will see this at the end of the notebook).

In this notebook we implement a simple version of k-means to quantize a grayscale image. Every pixel is a one-dimensional sample (its gray level), so the "distance" is just the absolute difference of two gray levels and the $K$ centroids form the palette of $K$ gray levels that describes the image best.

In [ ]:
import cv2
import math
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

In [ ]:
img = cv2.imread('data/kodim15.png')
img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img = img/255

### A short detour: broadcasting

The assignment step of k-means needs the distance of *every* pixel to *every* centroid. Instead of two nested loops, NumPy does it in a single line thanks to **broadcasting**: the rule that lets arrays of different shapes take part in one operation.

When two arrays are combined element by element, NumPy compares their shapes from the **last dimension backwards**. Two dimensions are compatible if they are equal or if one of them is 1 (a missing dimension counts as 1). The dimension of size 1 is then virtually *stretched* to match the other one; the data are not physically copied.

In [ ]:
a = np.array([[0, 10, 20],
              [30, 40, 50]])           # shape (2, 3)

print(a + 1)                           # a scalar is stretched to (2, 3)
print(a + np.array([1, 2, 3]))         # shape (3,) is repeated for every row
print(a + np.array([[100], [200]]))    # shape (2, 1) is repeated for every column

Now the k-means case in miniature: a tiny 2x3 "image" and $K = 3$ centroids. To get all the pixel-centroid distances we give the pixels an extra axis of size 1 with `np.newaxis`. The shapes then are (2, 3, 1) and (3,), which broadcast to (2, 3, 3): the last axis of the result runs over the centroids.

In [ ]:
pixels = np.array([[0.1, 0.5, 0.9],
                   [0.2, 0.4, 0.8]])   # shape (2, 3)
centroids = np.array([0.0, 0.5, 1.0])  # shape (3,), K = 3

diff = np.abs(pixels[..., np.newaxis] - centroids)
print(pixels[..., np.newaxis].shape, centroids.shape, '->', diff.shape)
print(diff[0, 1])                      # distances of the pixel [0, 1] (gray 0.5) to the three centroids
print(np.argmin(diff, axis=-1))        # index of the nearest centroid for every pixel

`diff[i, j, k]` is the distance between the pixel `[i, j]` and the centroid `k`. The `kmeans` function below spells the shape of the centroids out as `centroids[np.newaxis, np.newaxis, ...]`, i.e. (1, 1, K), while `quantize` leaves it to NumPy to add the missing leading dimensions; both give the same result.

The extra axis is not optional. Without it, `pixels - centroids` would silently compare the *columns* of the image with the centroids (the shapes (2, 3) and (3,) are compatible) and produce a meaningless result instead of an error. Shapes that are not compatible at all, like (4,) and (3,), raise `ValueError: operands could not be broadcast together`.

### Implementation of k-means iterations

The function below performs the two steps in a loop. The loop stops when the centroids do not change any more (or after `max_iter` iterations).

In [ ]:
def kmeans(img, centroids, max_iter=30):
    centroids = np.array(centroids, dtype=float)
    history = [np.copy(centroids)]
    for iteration in range(0, max_iter):
        # Cluster the pixels (i.e. for each pixel find the closest centroid)
        # We are taking advantage of python broadcasting here
        # In this case, we do not need to explicitly expand centroids with newaxis though
        diff = np.abs(img[..., np.newaxis] - centroids[np.newaxis, np.newaxis, ...])
        indices = np.argmin(diff, axis=-1)

        # After pixel clustering, recompute the centroids
        for ii in range(len(centroids)):
            centroids[ii] = np.mean(img[indices == ii])

        # Check if centroids have hit the convergence
        change = np.sum(np.abs(history[-1] - centroids))
        if change == 0:
            print('Convergence hit at iteration ', iteration)
            break

        # Update history (just for plotting purposes)
        history.append(np.copy(centroids))
    return centroids, history

We start with a deliberately poor initialization: all four centroids lie close to black. This way we can watch them travel to the right places.

In [ ]:
centroids, history = kmeans(img, [0, 0.1, 0.2, 0.3])
print('Centroids:', centroids)

We plot the history to visualize the evolution of centroid updates. Every line is one centroid.

In [ ]:
plt.plot(np.array(history))
plt.xlabel('Iteration')
plt.ylabel('Centroids')
plt.grid(True)

The brightest centroid settles quickly: it jumps from 0.3 to almost 0.9 in three steps. From the third iteration on no centroid moves by more than 0.05 (about 12 gray levels out of 255) in a single step, but the darker ones keep creeping: the third centroid slowly travels from about 0.26 to 0.45 during the remaining iterations. This slow tail is what makes the algorithm run for so many iterations.

### Quantization

After finding the centroids, i.e., the best fitting palette of gray levels, we quantize the original image: every pixel is replaced by its nearest palette entry. It is the very same assignment step as inside k-means, just performed once with the final centroids.

The result consists of two parts: the **palette** (`centroids`) and the **index map** that tells which palette entry every pixel uses. The quantized image is obtained by looking the indices up in the palette. This is how palette-based image formats store their data: for 4 colors a pixel index takes only 2 bits instead of 8.

Note that there is no loop over the pixels here, NumPy does the same work for all 393 216 pixels at once. A pixel-by-pixel Python loop gives the identical result but was about 80 times slower in our test (about 1 s instead of about 12 ms).

In [ ]:
def quantize(img, centroids):
    # Index of the nearest centroid for every pixel
    indices = np.argmin(np.abs(img[..., np.newaxis] - centroids), axis=-1)
    # Look the indices up in the palette
    return centroids[indices]

result = quantize(img, centroids)

In [ ]:
plt.subplot(121), plt.imshow(img, cmap='gray', vmin=0, vmax=1), plt.title('Original')
plt.subplot(122), plt.imshow(result, cmap='gray', vmin=0, vmax=1), plt.title('k-means, 4 gray levels')

### How good is the palette?

The lecture claims that k-means finds the colors that minimize the quantization error. Let's measure it. We use the PSNR from lesson 5 (the pixel values are in [0, 1] here, so the peak value is 1) and compare k-means with the **uniform quantization** with the same number of levels: 4 equally wide intensity intervals, each one reconstructed in its middle.

In [ ]:
def psnr(original, processed):
    return 10*np.log10(1/np.mean((original - processed)**2))

levels = 4
uniform = (np.minimum(np.floor(img*levels), levels - 1) + 0.5)/levels

plt.subplot(121), plt.imshow(uniform, cmap='gray', vmin=0, vmax=1)
plt.title(f'Uniform quantization, PSNR {psnr(img, uniform):.2f} dB')
plt.subplot(122), plt.imshow(result, cmap='gray', vmin=0, vmax=1)
plt.title(f'k-means, PSNR {psnr(img, result):.2f} dB')

With the same number of gray levels, k-means is about 1.5 dB better than the uniform quantization (its mean squared error is about 29 % lower). The uniform quantizer spreads its levels evenly over the intensity range, although about two thirds of the pixels of this photo are darker than mid-gray. K-means, on the other hand, puts three of its four palette entries into the dark half (about 0.12, 0.27 and 0.45, the uniform quantizer has two) and keeps just one for the bright pixels. It **adapts the palette to the image**, and that is exactly what minimizing the quantization error means.

### How many gray levels?

More clusters means a better approximation of the image. Let's run k-means for $K$ = 2, 4, 8 and 16, this time with the initial centroids spread evenly over the intensity range.

In [ ]:
for idx, K in enumerate([2, 4, 8, 16]):
    centroids_k, _ = kmeans(img, np.linspace(0.1, 0.9, K))
    quantized = quantize(img, centroids_k)
    plt.subplot(2, 2, idx + 1), plt.imshow(quantized, cmap='gray', vmin=0, vmax=1)
    plt.title(f'K = {K}, PSNR {psnr(img, quantized):.2f} dB')

Each doubling of the number of levels adds roughly 5-6 dB. Note that for $K = 4$ we now get 24.18 dB, slightly less than the 24.31 dB above, although it is the same problem. Only the initial centroids differ, which is the topic of the next section.

### Sensitivity to initialization

Since k-means finds only a local minimum, different starting points may end in different palettes. Let's run it several times from random initial centroids.

In [ ]:
np.random.seed(3)
for run in range(5):
    centroids_r, _ = kmeans(img, np.random.rand(4))
    print(f'Run {run}: palette {np.sort(centroids_r).round(3)}, PSNR {psnr(img, quantize(img, centroids_r)):.2f} dB')

We see that nothing guarantees we end up in the global minimum. The final palette depends on the (random) start.

### Summary

* K-means minimizes the sum of squared distances to the cluster centers by alternating the assignment and the update steps. It needs no labels (**unsupervised**).
* It converges to a local minimum that depends on the initialization. In practice it is therefore run several times and the best result is kept (e.g. the `attempts` parameter of OpenCV's `cv2.kmeans`), often with a smarter choice of the initial centroids (k-means++).
* The number of clusters $K$ is not found by the algorithm, we have to choose it.
* Here the samples were single gray levels. The same algorithm works for any samples for which a distance can be computed, e.g. RGB triplets. Note that k-means looks only at the values of the samples, not at where the pixel is in the image.